# 🔧 API Testing Notebook — OpenRouter, Groq & Google AI Studio

Tests all LLM providers **before** running the full Deep Agents app.  
Run each cell top-to-bottom. Green ✅ = working, Red ❌ = fix needed.

**Fallback logic:** OpenRouter (primary) → Groq (fallback) for Qwen 3.8 27B

## 1. Setup — Load Environment

In [4]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env from project root
load_dotenv(Path.cwd() / ".env")

# Check keys
keys = {
    "OPENROUTER_API_KEY": os.getenv("OPENROUTER_API_KEY", ""),
    "GROQ_API_KEY": os.getenv("GROQ_API_KEY", ""),
    "GOOGLE_API_KEY": os.getenv("GOOGLE_API_KEY", ""),
    "TAVILY_API_KEY": os.getenv("TAVILY_API_KEY", ""),
}

for name, val in keys.items():
    if not val or val.startswith("your_"):
        status = "❌ MISSING / PLACEHOLDER"
    else:
        status = f"✅ Set ({val[:12]}...)"
    print(f"{name}: {status}")

OPENROUTER_API_KEY: ✅ Set (sk-or-v1-c05...)
GROQ_API_KEY: ✅ Set (gsk_ij15LwaO...)
GOOGLE_API_KEY: ✅ Set (AQ.Ab8RN6I8y...)
TAVILY_API_KEY: ✅ Set (tvly-dev-1MP...)


---
## 2. Test OpenRouter — Raw HTTP

Tests the OpenRouter API directly. If this fails, the Groq fallback should kick in.

In [16]:
import requests

OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY", "")
GROQ_KEY = os.getenv("GROQ_API_KEY", "")

def test_openrouter_raw(model_id: str, prompt: str = "Say hello in one sentence."):
    """Test OpenRouter API with a raw HTTP POST."""
    print(f"\n{'='*60}")
    print(f"Testing OpenRouter: {model_id}")
    print(f"{'='*60}")
    
    try:
        resp = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_KEY}",
                "Content-Type": "application/json",
            },
            json={
                "model": model_id,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 100,
            },
            timeout=30,
        )
        
        print(f"Status Code: {resp.status_code}")
        
        if resp.status_code == 200:
            data = resp.json()
            content = data["choices"][0]["message"]["content"]
            model_used = data.get("model", "unknown")
            usage = data.get("usage", {})
            print(f"✅ SUCCESS")
            print(f"Model Used: {model_used}")
            print(f"Tokens: {usage.get('prompt_tokens', '?')} in / {usage.get('completion_tokens', '?')} out")
            print(f"Response: {content[:200]}")
            return True
        else:
            print(f"❌ FAILED")
            print(f"Response: {resp.text[:500]}")
            if resp.status_code == 401:
                print("\n🔑 FIX: OPENROUTER_API_KEY is invalid. Get a new one at https://openrouter.ai/settings/keys")
            elif resp.status_code == 402:
                print("\n💳 FIX: No credits. Use a :free model or add credits.")
            elif resp.status_code == 404:
                print(f"\n🔍 FIX: Model '{model_id}' not found. Check https://openrouter.ai/models")
            elif resp.status_code == 429:
                print("\n⏱️ FIX: Rate limited. Wait a moment and retry.")
            return False
    except Exception as e:
        print(f"❌ ERROR: {e}")
        return False

qwen_or_ok = test_openrouter_raw("qwen/qwen3.8-27b:free")


Testing OpenRouter: qwen/qwen3.8-27b:free
Status Code: 429
❌ FAILED
Response: {"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"qwen/qwen3.8-27b:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations","provider_name":"ModelRun","is_byok":false,"provider_error_code":"429","limit_source":"upstream_provider_shared_pool","remedy_hint":"Retry shortly, add your own provider key (https://openrouter.ai/settings/integrations), or route to another prov

⏱️ FIX: Rate limited. Wait a moment and retry.


---
## 3. Test Groq — Raw HTTP (Fallback)

Tests Groq API directly. This is the fallback when OpenRouter is down.

In [6]:
def test_groq_raw(model_id: str, prompt: str = "Say hello in one sentence."):
    """Test Groq API with a raw HTTP POST."""
    print(f"\n{'='*60}")
    print(f"Testing Groq: {model_id}")
    print(f"{'='*60}")
    
    if not GROQ_KEY or GROQ_KEY.startswith("your_"):
        print("⚠️ GROQ_API_KEY not set — skipping")
        print("   Get a key at: https://console.groq.com/keys")
        return None  # None = skipped
    
    try:
        resp = requests.post(
            url="https://api.groq.com/openai/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {GROQ_KEY}",
                "Content-Type": "application/json",
            },
            json={
                "model": model_id,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 100,
            },
            timeout=30,
        )
        
        print(f"Status Code: {resp.status_code}")
        
        if resp.status_code == 200:
            data = resp.json()
            content = data["choices"][0]["message"]["content"]
            model_used = data.get("model", "unknown")
            usage = data.get("usage", {})
            print(f"✅ SUCCESS")
            print(f"Model Used: {model_used}")
            print(f"Tokens: {usage.get('prompt_tokens', '?')} in / {usage.get('completion_tokens', '?')} out")
            print(f"Response: {content[:200]}")
            return True
        else:
            print(f"❌ FAILED")
            print(f"Response: {resp.text[:500]}")
            if resp.status_code == 401:
                print("\n🔑 FIX: GROQ_API_KEY is invalid. Get a new one at https://console.groq.com/keys")
            elif resp.status_code == 404:
                print(f"\n🔍 FIX: Model '{model_id}' not found on Groq.")
            return False
    except Exception as e:
        print(f"❌ ERROR: {e}")
        return False

qwen_groq_ok = test_groq_raw("qwen/qwen3.8-27b")


Testing Groq: qwen/qwen3.8-27b
Status Code: 200
✅ SUCCESS
Model Used: qwen/qwen3.8-27b
Tokens: 18 in / 3 out
Response: Hello!


---
## 4. Test LangChain Integration — OpenRouter + Groq

In [7]:
# Check imports
imports_ok = True
for pkg, cls in [("langchain_openrouter", "ChatOpenRouter"), ("langchain_groq", "ChatGroq"), ("langchain_google_genai", "ChatGoogleGenerativeAI")]:
    try:
        mod = __import__(pkg, fromlist=[cls])
        print(f"✅ {pkg}.{cls} imported")
    except ImportError as e:
        print(f"❌ {pkg} — {e}")
        print(f"   FIX: pip install -U {pkg.replace('_', '-')}")
        imports_ok = False

✅ langchain_openrouter.ChatOpenRouter imported
✅ langchain_groq.ChatGroq imported
✅ langchain_google_genai.ChatGoogleGenerativeAI imported


In [8]:
from langchain.chat_models import init_chat_model

def test_init_chat_model(label: str, model_string: str):
    """Test the provider:model format used by deepagents."""
    print(f"\nTesting: {label}")
    print(f"  init_chat_model('{model_string}')")
    try:
        llm = init_chat_model(model_string, temperature=0, max_tokens=100)
        response = llm.invoke("What is the capital of France? One word.")
        print(f"  ✅ SUCCESS: {response.content[:200]}")
        print(f"     Model class: {type(llm).__name__}")
        return True
    except Exception as e:
        print(f"  ❌ FAILED: {e}")
        return False

# Test all model strings from config.py
lc_results = {}

lc_results["OpenRouter Qwen"] = test_init_chat_model(
    "OpenRouter Qwen 3.8 27B", "openrouter:qwen/qwen3.8-27b:free"
)

if GROQ_KEY and not GROQ_KEY.startswith("your_"):
    lc_results["Groq Qwen"] = test_init_chat_model(
        "Groq Qwen 3.8 27B (fallback)", "groq:qwen/qwen3.8-27b"
    )
else:
    print("\n⚠️ Skipping Groq LangChain test — GROQ_API_KEY not set")
    lc_results["Groq Qwen"] = None

print("\n" + "="*60)
for name, ok in lc_results.items():
    if ok is None:
        print(f"  ⚠️  {name}: SKIPPED (no API key)")
    else:
        print(f"  {'✅' if ok else '❌'}  {name}: {'PASS' if ok else 'FAIL'}")
print("="*60)


Testing: OpenRouter Qwen 3.8 27B
  init_chat_model('openrouter:qwen/qwen3.8-27b:free')
  ❌ FAILED: Provider returned error

Testing: Groq Qwen 3.8 27B (fallback)
  init_chat_model('groq:qwen/qwen3.8-27b')
  ✅ SUCCESS: Paris
     Model class: ChatGroq

  ❌  OpenRouter Qwen: FAIL
  ✅  Groq Qwen: PASS


---
## 5. Test Google AI Studio Models

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

def test_google_model(model_id: str):
    """Test Google AI Studio model."""
    print(f"\nTesting Google AI: {model_id}")
    try:
        llm = ChatGoogleGenerativeAI(
            model=model_id,
            temperature=0,
            max_output_tokens=100,
        )
        response = llm.invoke("What is 3+3? Answer in one word.")
        print(f"  ✅ SUCCESS: {response.content[:200]}")
        return True
    except Exception as e:
        print(f"  ❌ FAILED: {e}")
        if "API_KEY" in str(e).upper() or "401" in str(e):
            print("  🔑 FIX: Check GOOGLE_API_KEY at https://aistudio.google.dev/")
        return False

gemini_ok = test_google_model("gemini-3.8-flash")
gemma_ok = test_google_model("gemma-4-31b-it")

print(f"\nGoogle: Gemini = {'✅' if gemini_ok else '❌'}  |  Gemma = {'✅' if gemma_ok else '❌'}")


Testing Google AI: gemini-3.8-flash


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


  ✅ SUCCESS: [{'type': 'text', 'text': 'Six', 'extras': {'signature': 'EtMCCtACAWkUfRNYmwTetxm/D4KP79VfH2RUcRpNgiDAwjC3dcWzOK5YGDG2EiFGDoPsOfvjWRCt4vlXRq3lTZzMlH3Iu6aqRuuoo9itWF6naaLkS4Rufi/XV02k/OU9Lcs5rMPF8/bfJMzaMFvxXbhjsTUEXLVzlYo2uMdr/hOEEY3nHcgXM59PotF1YvAA4eLvCkgowpe0wa/DwN2yCtGss68b6vsqS46qmlqkF16e3G6rsSp8+No2SaTHsEx1t8Ebcoh3ddFvjmo2nwnZjArCE3zbOc21/Gtxv6YmCKBtW+frlIvwl/YFQTYWuPxmXRMo9VaIupppGje3eNyz6PbHahZQsUY06LOc0YwEmQ0xPeAvu31RAaxGD/X/Bwnfv4RoH10ds44rlRaOqBS2pHprqFGkMkhWF1bQfyxz3qVcQTQe51LE2dNFAjRoU4Z25Qus6BDS'}}]

Testing Google AI: gemma-4-31b-it
  ✅ SUCCESS: [{'type': 'thinking', 'thinking': 'The user is asking for the sum of 3 + 3.\nThe constraint is to answer in "one word".\n\nCalculation: 3 + 3 = 6.\nOne word answer: "Six" or "6". Usually, "Six" is a word, "6" is a digit. Both are acceptable as single-token/single-word responses.\n\nFinal Answer: Six.'}, {'type': 'text', 'text': 'Six'}]

Google: Gemini = ✅  |  Gemma = ✅


---
## 6. Test Tavily Search

In [10]:
from tavily import TavilyClient

try:
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY", ""))
    result = client.search("test query", max_results=1)
    print(f"✅ Tavily working — got {len(result.get('results', []))} result(s)")
    tavily_ok = True
except Exception as e:
    print(f"❌ Tavily failed: {e}")
    tavily_ok = False

✅ Tavily working — got 1 result(s)


---
## 7. Test Fallback Logic

Tests the `resolve_model_id()` function from `config.py`.

In [11]:
# Add src to path
src = str(Path.cwd() / "src")
if src not in sys.path:
    sys.path.insert(0, src)

from deep_agents.config import resolve_model_id, FALLBACK_MAP, _is_openrouter_available

print("Fallback Map:")
for primary, fallback in FALLBACK_MAP.items():
    print(f"  {primary}  →  {fallback}")

print(f"\nOpenRouter Available: {_is_openrouter_available()}")

# Test resolution
test_ids = [
    "openrouter:qwen/qwen3.8-27b:free",
    "google_genai:gemini-3.8-flash",
    "google_genai:gemma-4-31b-it",
]

print("\nModel Resolution:")
for mid in test_ids:
    resolved = resolve_model_id(mid)
    if resolved != mid:
        print(f"  🔄 {mid}  →  {resolved}  (FALLBACK)")
    else:
        print(f"  ✅ {mid}  (unchanged)")

Fallback Map:
  openrouter:qwen/qwen3.8-27b:free  →  groq:qwen/qwen3.8-27b

OpenRouter Available: True

Model Resolution:
  ✅ openrouter:qwen/qwen3.8-27b:free  (unchanged)
  ✅ google_genai:gemini-3.8-flash  (unchanged)
  ✅ google_genai:gemma-4-31b-it  (unchanged)


---
## 8. Full Summary

In [12]:
print("\n" + "="*60)
print("  🧪 FULL API TEST RESULTS")
print("=" * 60)

all_results = {
    "OpenRouter — Qwen 3.8 27B (primary)": qwen_or_ok,
    "Groq — Qwen 3.8 27B (fallback)": qwen_groq_ok,
    "Google — Gemini 3.8 Flash": gemini_ok,
    "Google — Gemma 4 31B": gemma_ok,
    "Tavily — Web Search": tavily_ok,
}

print()
for name, ok in all_results.items():
    if ok is None:
        print(f"  ⚠️  {name}: SKIPPED")
    else:
        print(f"  {'✅' if ok else '❌'}  {name}")

# Evaluate readiness
qwen_available = qwen_or_ok or (qwen_groq_ok is True)
google_available = gemini_ok or gemma_ok

print(f"\n  {'─'*40}")
print(f"  Qwen (any provider):   {'✅ READY' if qwen_available else '❌ NO QWEN AVAILABLE'}")
print(f"  Google Models:         {'✅ READY' if google_available else '❌ NO GOOGLE AVAILABLE'}")
print(f"  Web Search:            {'✅ READY' if tavily_ok else '❌ SEARCH UNAVAILABLE'}")

if qwen_available and google_available:
    print("\n  🎉 Ready to run: streamlit run app.py")
    if not qwen_or_ok and qwen_groq_ok:
        print("  ℹ️  OpenRouter is down — Groq fallback will be used automatically.")
else:
    print("\n  ⚠️ Fix failing tests before running the app.")
    print("  API Key Sources:")
    print("    OpenRouter: https://openrouter.ai/settings/keys")
    print("    Groq:       https://console.groq.com/keys")
    print("    Google:     https://aistudio.google.dev/")
    print("    Tavily:     https://tavily.com/")


  🧪 FULL API TEST RESULTS

  ❌  OpenRouter — Qwen 3.8 27B (primary)
  ✅  Groq — Qwen 3.8 27B (fallback)
  ✅  Google — Gemini 3.8 Flash
  ✅  Google — Gemma 4 31B
  ✅  Tavily — Web Search

  ────────────────────────────────────────
  Qwen (any provider):   ✅ READY
  Google Models:         ✅ READY
  Web Search:            ✅ READY

  🎉 Ready to run: streamlit run app.py
  ℹ️  OpenRouter is down — Groq fallback will be used automatically.


---
## 9. Debug — Check OpenRouter Account

In [13]:
try:
    resp = requests.get(
        "https://openrouter.ai/api/v1/auth/key",
        headers={"Authorization": f"Bearer {OPENROUTER_KEY}"},
        timeout=10,
    )
    if resp.status_code == 200:
        data = resp.json().get("data", {})
        print(f"✅ API Key is valid")
        print(f"   Label: {data.get('label', 'N/A')}")
        print(f"   Credits: ${data.get('limit', 'N/A')}")
        print(f"   Usage: ${data.get('usage', 'N/A')}")
        remaining = (data.get('limit', 0) or 0) - (data.get('usage', 0) or 0)
        print(f"   Remaining: ${remaining:.4f}")
    elif resp.status_code == 401:
        print("❌ API Key is INVALID")
        print("   Get a new key: https://openrouter.ai/settings/keys")
    else:
        print(f"⚠️ Status: {resp.status_code} — {resp.text[:300]}")
except Exception as e:
    print(f"❌ Could not check account: {e}")

✅ API Key is valid
   Label: sk-or-v1-c05...d9d
   Credits: $None
   Usage: $0
   Remaining: $0.0000


In [14]:
# Verify our model exists on OpenRouter
try:
    resp = requests.get("https://openrouter.ai/api/v1/models", headers={"Authorization": f"Bearer {OPENROUTER_KEY}"}, timeout=15)
    if resp.status_code == 200:
        models = resp.json().get("data", [])
        all_ids = {m["id"] for m in models}
        target = "qwen/qwen3.8-27b:free"
        print(f"Total models on OpenRouter: {len(models)}")
        if target in all_ids:
            print(f"✅ {target} — EXISTS")
        else:
            print(f"❌ {target} — NOT FOUND")
            similar = [m["id"] for m in models if "qwen" in m.get("id", "").lower()][:10]
            if similar:
                print("   Similar Qwen models:")
                for s in similar:
                    print(f"     • {s}")
except Exception as e:
    print(f"❌ Could not fetch model list: {e}")

Total models on OpenRouter: 458
✅ qwen/qwen3.8-27b:free — EXISTS
